<a href="https://colab.research.google.com/github/herrypu/HP_LLM4CD_S6SP/blob/main/chipchat_exampleB_i2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM4ChipDesign HW1 Chipchat
## Example B (Example 2: Sequence Detector)


In [ ]:
API_KEY=""
MAX_TOKENS=4068
TEMPERATURE=0.7

### 1. Setting up

In [2]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [3]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = API_KEY,
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )
    content = completion.choices[0].message.content

    print("-" * 30 + " LLM Response " + "-" * 30)
    print(content)
    print("-" * 74)

    return content

### 2. Verilog Generation

In [4]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   4630      0 --:--:-- --:--:-- --:--:--  4635


In [5]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (3 bits)
	- Outputs:
		- Sequence found

While enabled, it should detect the following sequence of binary input values:
	- 0b001
	- 0b101
	- 0b110
	- 0b000
	- 0b110
	- 0b110
	- 0b011
	- 0b101

How would I write a design that meets these specifications?
'''

In [6]:
ask_deepseek(verilog_generation_prompt)

------------------------------ LLM Response ------------------------------
Here's a Verilog model for your sequence detector:

```verilog
module sequence_detector(
    input clk,
    input rst_n,        // Active-low reset
    input [2:0] data,   // 3-bit input
    output reg seq_found
);
    
    // State definitions
    localparam  IDLE    = 4'd0,
                S1_001  = 4'd1,
                S2_101  = 4'd2,
                S3_110  = 4'd3,
                S4_000  = 4'd4,
                S5_110  = 4'd5,
                S6_110  = 4'd6,
                S7_011  = 4'd7,
                S8_101  = 4'd8;
    
    reg [3:0] state, next_state;
    
    // State register
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            state <= IDLE;
            seq_found <= 1'b0;
        end else begin
            state <= next_state;
            // Assert seq_found only when we reach the final state
            seq_found <= (next_state == S8_101);
        end
    end
  

"Here's a Verilog model for your sequence detector:\n\n```verilog\nmodule sequence_detector(\n    input clk,\n    input rst_n,        // Active-low reset\n    input [2:0] data,   // 3-bit input\n    output reg seq_found\n);\n    \n    // State definitions\n    localparam  IDLE    = 4'd0,\n                S1_001  = 4'd1,\n                S2_101  = 4'd2,\n                S3_110  = 4'd3,\n                S4_000  = 4'd4,\n                S5_110  = 4'd5,\n                S6_110  = 4'd6,\n                S7_011  = 4'd7,\n                S8_101  = 4'd8;\n    \n    reg [3:0] state, next_state;\n    \n    // State register\n    always @(posedge clk or negedge rst_n) begin\n        if (!rst_n) begin\n            state <= IDLE;\n            seq_found <= 1'b0;\n        end else begin\n            state <= next_state;\n            // Assert seq_found only when we reach the final state\n            seq_found <= (next_state == S8_101);\n        end\n    end\n    \n    // Next state logic\n    always 

### 3. Extraction and Verification

In [11]:
output_verilog_code = '''
module sequence_detector(
    input clk,
    input rst_n,        // Active-low reset
    input [2:0] data,   // 3-bit input
    output reg seq_found
);

    // State definitions
    localparam  IDLE    = 4'd0,
                S1_001  = 4'd1,
                S2_101  = 4'd2,
                S3_110  = 4'd3,
                S4_000  = 4'd4,
                S5_110  = 4'd5,
                S6_110  = 4'd6,
                S7_011  = 4'd7,
                S8_101  = 4'd8;

    reg [3:0] state, next_state;

    // State register
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            state <= IDLE;
            seq_found <= 1'b0;
        end else begin
            state <= next_state;
            // Assert seq_found only when we reach the final state
            seq_found <= (next_state == S8_101);
        end
    end

    // Next state logic
    always @(*) begin
        case (state)
            IDLE: begin
                if (data == 3'b001)
                    next_state = S1_001;
                else
                    next_state = IDLE;
            end

            S1_001: begin
                if (data == 3'b101)
                    next_state = S2_101;
                else if (data == 3'b001)
                    next_state = S1_001;  // Reset to beginning if first value repeats
                else
                    next_state = IDLE;
            end

            S2_101: begin
                if (data == 3'b110)
                    next_state = S3_110;
                else if (data == 3'b001)
                    next_state = S1_001;  // Partial reset
                else
                    next_state = IDLE;
            end

            S3_110: begin
                if (data == 3'b000)
                    next_state = S4_000;
                else if (data == 3'b001)
                    next_state = S1_001;  // Partial reset
                else
                    next_state = IDLE;
            end

            S4_000: begin
                if (data == 3'b110)
                    next_state = S5_110;
                else if (data == 3'b001)
                    next_state = S1_001;  // Partial reset
                else
                    next_state = IDLE;
            end

            S5_110: begin
                if (data == 3'b110)
                    next_state = S6_110;
                else if (data == 3'b001)
                    next_state = S1_001;  // Partial reset
                else
                    next_state = IDLE;
            end

            S6_110: begin
                if (data == 3'b011)
                    next_state = S7_011;
                else if (data == 3'b001)
                    next_state = S1_001;  // Partial reset
                else
                    next_state = IDLE;
            end

            S7_011: begin
                if (data == 3'b101)
                    next_state = S8_101;
                else if (data == 3'b001)
                    next_state = S1_001;  // Partial reset
                else
                    next_state = IDLE;
            end

            S8_101: begin
                // After complete sequence, look for next sequence start
                if (data == 3'b001)
                    next_state = S1_001;
                else
                    next_state = IDLE;
            end

            default: next_state = IDLE;
        endcase
    end
endmodule
'''
filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [12]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

sequence_detector_tb.v:10: error: port ``reset_n'' is not a port of dut.
sequence_detector_tb.v:10: error: port ``sequence_found'' is not a port of dut.
2 error(s) during elaboration.
